# Stage 2 Notebook 60 - Exp2EEE Anchor + VFL + full 70K + bb_throttle + 12 epochs

**Push anchor geometry past 0.60.** NB48 (6 ep, full data) hit matched_iou=0.544. NB57 (6 ep + bb_throttle=0.01) hit 0.553 (project record). Both showed matched_iou still climbing at the final epoch -- the runs were too short. Exp2EEE extends to 12 epochs at full data (~105,000 iterations total, 2x NB48/NB57).

Same as NB57 but `end_epoch: 6 -> 12`. If matched_iou pushes past 0.60, that's geometry headroom that puts us in CLRKDNet-CULane competitive territory.

If matched_iou plateaus at ~0.56: the geometry is capacity-bound and we need a wider backbone (NB45 used width 1.0 + 30 ep at limit=3000 and hit 0.525). Combining width=1.0 with full data + 12 epochs would be the next escalation.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~2.5-3 hr wall-clock.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint_smoke.log
OK exp55_rmt_gca_anchor_vfl_full_data_long12_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.3711 det_loss=3.9906 grad_cos=0.2592 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49957242608070374, 'gate/lane_mean': 0.5015766024589539, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp55_rmt_gca_anchor_vfl_full_data_long12_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp55_rmt_gca_anchor_vfl_full_data_long12_joint.y

0

## What to watch in Exp2EEE

Reference NB57 (6 ep full + bb_throttle): matched_iou=0.553, oracle_f1=0.455, decoded_f1=0.042, val_det=3.17.

Pass criteria at epoch 12:
- **val/matched_line_iou >= 0.60** -- decisive geometry improvement; matches CLRKDNet on CULane.
- val/lane/decoded_oracle_f1 >= 0.50.
- val/lane/decoded_f1 >= 0.05 (don't regress NB48).
- train_lane still decreasing at epoch 12 OR has plateaued (use plateau as signal to stop).

If matched_iou >= 0.60: the geometry breakthrough. We have a 'super geometry' model.
Combine with NB59's strong-cls model in Exp2FFF (post-hoc cls re-ranking by feeding NB59's trained query head as a re-ranker over NB60's anchor curves).